# Model comparison

Minimal notebook to compare the final Negative Binomial model against a Poisson baseline with a strict temporal split.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance, mean_squared_error

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Open this notebook from the repository root or from the analysis/ directory.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.model import build_final_parquet, build_training_xy, train_model

ACCIDENTS_CSV_PATH = r"REPLACE_WITH_ACCIDENTS_CSV"
NETWORK_ZIP_PATH = None
OUTPUT_PARQUET_PATH = REPO_ROOT / "outputs/modeling/training_table_with_exogenous_context_features.parquet"
FORCE_REBUILD = False


def predict_statsmodels(result, X):
    X = pd.DataFrame(X).astype(float).copy()
    expected_columns = list(result.model.exog_names)

    if "const" in expected_columns and "const" not in X.columns:
        X["const"] = 1.0

    missing_columns = [column for column in expected_columns if column not in X.columns]
    if missing_columns:
        raise ValueError(f"Missing columns for prediction: {missing_columns}")

    X = X[expected_columns]
    y_pred = np.asarray(result.predict(X), dtype=float)
    y_pred = np.nan_to_num(y_pred, nan=1e-9, posinf=1e9, neginf=1e-9)
    return np.clip(y_pred, 1e-9, None)


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 1e-9, None)
    return {
        "mean_poisson_deviance": float(mean_poisson_deviance(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


In [ ]:
if "REPLACE_WITH" in str(ACCIDENTS_CSV_PATH):
    raise ValueError("Set ACCIDENTS_CSV_PATH in the first cell before running the notebook.")

accidents_csv_path = Path(ACCIDENTS_CSV_PATH).expanduser()
network_zip_path = None if NETWORK_ZIP_PATH in (None, "") else Path(NETWORK_ZIP_PATH).expanduser()

parquet_path = build_final_parquet(
    accidents_csv_path=accidents_csv_path,
    output_parquet_path=OUTPUT_PARQUET_PATH,
    network_zip_path=network_zip_path,
    force=FORCE_REBUILD,
)

panel = pd.read_parquet(parquet_path)
if "analysis_year" not in panel.columns:
    raise ValueError("Missing required column 'analysis_year' in the parquet.")
if panel["analysis_year"].isna().any():
    raise ValueError("Column 'analysis_year' contains null values.")

analysis_year = pd.to_numeric(panel["analysis_year"], errors="raise").astype(int)
available_years = sorted(analysis_year.unique().tolist())
train_years = [year for year in available_years if 2016 <= year <= 2022]
required_eval_years = [2023, 2024]
missing_eval_years = [year for year in required_eval_years if year not in available_years]

if not train_years:
    raise ValueError("The parquet does not contain training rows for the 2016-2022 period.")
if missing_eval_years:
    raise ValueError(f"The parquet is missing evaluation years: {missing_eval_years}")

X, y = build_training_xy(parquet_path, include_analysis_year=False)
if not (len(X) == len(y) == len(analysis_year)):
    raise ValueError("Parquet rows and model matrices are misaligned.")

print(f"Parquet ready at: {parquet_path}")
print(f"Available years: {available_years}")
print(f"Rows available for modeling: {len(X)}")


In [ ]:
train_mask = (analysis_year <= 2022).to_numpy()
validation_mask = (analysis_year == 2023).to_numpy()
test_mask = (analysis_year == 2024).to_numpy()

if not train_mask.any():
    raise ValueError("The train split is empty.")
if not validation_mask.any():
    raise ValueError("The validation split is empty.")
if not test_mask.any():
    raise ValueError("The test split is empty.")

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()
X_validation = X.loc[validation_mask].copy()
y_validation = y.loc[validation_mask].copy()
X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

split_sizes = pd.Series(
    {
        "train_rows": int(train_mask.sum()),
        "validation_rows": int(validation_mask.sum()),
        "test_rows": int(test_mask.sum()),
    }
)
split_sizes


In [ ]:
negative_binomial = train_model(X_train, y_train)

X_train_poisson = sm.add_constant(pd.DataFrame(X_train).astype(float), has_constant="add")
y_train_poisson = pd.Series(y_train).astype(float)
poisson = sm.GLM(y_train_poisson, X_train_poisson, family=sm.families.Poisson()).fit()

print("Models fitted: negative_binomial, poisson")


In [ ]:
rows = {}
splits = {
    "validation": (X_validation, y_validation),
    "test": (X_test, y_test),
}
models = {
    "negative_binomial": negative_binomial,
    "poisson": poisson,
}

for model_name, result in models.items():
    row = {}
    for split_name, (X_split, y_split) in splits.items():
        y_pred = predict_statsmodels(result, X_split)
        metrics = compute_metrics(y_split, y_pred)
        for metric_name, metric_value in metrics.items():
            row[f"{split_name}_{metric_name}"] = metric_value
    rows[model_name] = row

results = pd.DataFrame.from_dict(rows, orient="index")
ordered_columns = [
    "validation_mean_poisson_deviance",
    "validation_mae",
    "validation_rmse",
    "test_mean_poisson_deviance",
    "test_mae",
    "test_rmse",
]
results = results[ordered_columns]
results.round(6)
